# Mash Plus V4 — independent geographic satellite experiment

**Research only.** This is a new, preregistered experiment, not a repeat of the failed V3 Hammar test. It uses ESA Sentinel-2 L2A from Microsoft Planetary Computer and trains two reviewed architectures: our ConvGRU persistence residual and the upstream open-source [segmentation_models_pytorch U-Net](https://github.com/qubvel-org/segmentation_models.pytorch) (package, not copied source). The original Hammar test has already been inspected and is **not used as the final independent V4 test**.

- **Training**: three pilot rectangles at Hawizeh, Central Marshes, and Sinnaf seasonal wetlands. **Completely new spatial test**: Dalmaj Marsh, never used by model fitting/normalization/validation. The Dalmaj pilot center is sourced from [Key Biodiversity Areas Dalmaj 8043](https://www.keybiodiversityareas.org/en/site/factsheet/8043), and Sinnaf from [KBA 8050](https://www.keybiodiversityareas.org/en/site/factsheet/8050).
- 48 monthly periods from October 2022 through September 2026; pilot rectangles are **not protected-area boundaries**, and the 64×64 grid is coarse, not Sentinel 10-meter native resolution.
- Train-only target-aware shoreline sampling and wet-class weighting. **Never resample held-out validation/test pixels.** Both alternatives are chosen exclusively by chronological validation. The Dalmaj test is reported once, not optimized against.
- Labels are unverified MNDWI/NDVI proxy, **not ground-truth hydrology**. A 2026 Landsat-8/9 cross-sensor audit is supplementary optical evidence, not human truth. Training may reject both networks and choose a simple baseline.

Run with Kaggle GPU and Internet enabled. Outputs must be retained even on negative findings. No field decisions or model certification are authorized by this notebook.

In [ ]:
# Install necessary free software packages; upstream libraries retain their licenses.
%pip -q install "numpy>=1.26,<3" "rasterio>=1.3,<2" "pystac-client>=0.8,<1" "planetary-computer>=1.0,<2" "segmentation-models-pytorch>=0.4,<1"
import os,sys,subprocess,json,hashlib,zipfile,traceback
from pathlib import Path
import numpy as np
import torch
print("PYTHON",sys.version.split()[0],"TORCH",torch.__version__)
print("CUDA",torch.cuda.is_available(),"GPU",torch.cuda.get_device_name(0) if torch.cuda.is_available() else "MISSING")
assert torch.cuda.is_available(),"A genuine V4 GPU experiment requires Kaggle accelerator enabled"

## 1 — Version-locked executable source and independent geographic protocol

The notebook clones the public repository and records the exact Git commit SHA. All models, acquisition manifests and predictions are stored in the output, not substituted with sample values.

In [ ]:
ROOT=Path("/kaggle/working/MarshPulse")
if not (ROOT/"ml/train_v3.py").is_file():
    subprocess.run(["git","clone","--depth","1","https://github.com/ahpa5246-lgtm/MarshPulse.git",str(ROOT)],check=True)
COMMIT=subprocess.check_output(["git","-C",str(ROOT),"rev-parse","HEAD"],text=True).strip()
for needed in ("ml/collect_sentinel.py","ml/train_v3.py","ml/mash_plus_smp.py","ml/landsat_crosscheck.py","ml/judge_readiness.py","ml/prepare_field_review.py"):
    assert (ROOT/needed).is_file(),f"Missing real implementation: {needed}"
OUT=Path("/kaggle/working/mash_plus_v4_evidence")
OUT.mkdir(parents=True,exist_ok=True)
(OUT/"source_commit.txt").write_text(COMMIT+"\n")
print("SOURCE_COMMIT",COMMIT)

## 2 — Acquire true Sentinel-2 scenes from three training pilots and sealed Dalmaj holdout

The coordinates below were specified **before** the Dalmaj imagery was acquired or any V4 model was evaluated. Source: KBA 8043 Dalmaj (32.2000, 45.4667), KBA 8050 Sinnaf (31.8667,47.3167), KBA 8056 Hawizeh (31.5772,47.6848). Do not shift the holdout box after seeing performance. A coverage failure means the study is inconclusive.

In [ ]:
AOIS={
    "hawizeh_train":[47.64,31.53,47.74,31.63],
    "central_train":[47.00,31.04,47.10,31.14],
    "sinnaf_train":[47.2667,31.8167,47.3667,31.9167],
    "dalmaj_spatial_test":[45.4167,32.15,45.5167,32.25],
}
START,END="2022-10","2026-09"
GRID_PIXELS=64
MAX_SCENES=2
IMAGE_ROOT=OUT/"source_imagery"
PATHS={}
for name,bbox in AOIS.items():
    destination=IMAGE_ROOT/name
    command=[sys.executable,str(ROOT/"ml/collect_sentinel.py"),
             "--bbox",*map(str,bbox),"--start",START,"--end",END,
             "--pixels",str(GRID_PIXELS),"--max-scenes",str(MAX_SCENES),
             "--min-coverage",".2","--area-label",name+" KBA pilot, not official site boundary",
             "--out",str(destination)]
    print("COLLECTING_REAL_SENTINEL",name,bbox,flush=True)
    subprocess.run(command,check=True)
    PATHS[name]=destination/"hawizeh_s2_monthly.npz"
print("COLLECTED_REGIONS",list(PATHS))

## 3 — Data audit and training-only class imbalance checks

Reject fabricated grids, incomplete source records and missing coverage. Water prevalence and class support are **reported separately**; do not choose the holdout AOI or threshold based on its wet fraction.

In [ ]:
QUALITY={}
for name,npz in PATHS.items():
    output=OUT/("quality_"+name+".json")
    subprocess.run([sys.executable,str(ROOT/"ml/quality_gate.py"),"--input",str(npz),"--out",str(output)],check=True)
    quality=json.loads(output.read_text())
    manifest=json.loads((npz.parent/"satellite_manifest.json").read_text())
    assert len(manifest["months"])==48
    assert all(m["scenes"] for m in manifest["months"] if m["fraction_valid"]>0)
    QUALITY[name]=quality
    print("REGION_AUDIT",name,"wet_proxy_pixels",quality["total_proxy_wet_pixels"],
          "wet_months",quality["months_with_wet_proxy"],
          "water_prevalence",quality["overall_wet_fraction"],
          "warnings",quality["warnings"])
assert all(q["total_observed_pixels"]>0 for q in QUALITY.values()),"No real observed pixels"

## 4 — Execute both predeclared architectures (no hyperparameter search on Dalmaj)

The frozen alternatives use identical AOIs, date windows and chronological splits. Both methods start with last-observation persistence and learn only residual corrections. Extra wet/transition month sampling, water-centered 32-pixel training crops and BCE weighting apply only to **training**. Validation and Dalmaj test remain natural and unresampled.

We evaluate both predeclared candidates and determine any winner by **validation-only** criteria, before consulting Dalmaj metrics. Any performance claim must disclose that two architectures were compared; this is a research screen, not a formal significance test.

In [ ]:
COMMON=[sys.executable,str(ROOT/"ml/train_v3.py"),
    "--train-input",str(PATHS["hawizeh_train"]),
    "--train-input",str(PATHS["central_train"]),
    "--train-input",str(PATHS["sinnaf_train"]),
    "--spatial-test-input",str(PATHS["dalmaj_spatial_test"]),
    "--v4","--batch","2","--lookback","4","--seeds","17,42",
    "--min-wet-valid","25","--wet-bce-ratio","3",
    "--wet-sampler-boost","3","--change-sampler-boost","3",
    "--crop-size","32","--water-center-prob",".75"]
TRIALS=[
    ("convgru",["--architecture","convgru","--epochs","20","--patience","6","--base","16","--lr",".00015"]),
    ("smp-unet",["--architecture","smp-unet","--encoder","resnet18",
                 "--epochs","18","--patience","6","--base","16","--lr",".00015"]),
]
REPORTS={}
for architecture,flags in TRIALS:
    result_dir=OUT/"evaluations"/architecture
    print("TRAINING_REAL_GPU_ARCHITECTURE",architecture,flush=True)
    subprocess.run([*COMMON,*flags,"--out",str(result_dir)],check=True)
    report=json.loads((result_dir/"v3_evaluation.json").read_text())
    assert report["device"].startswith("cuda"),"GPU training unexpectedly fell back to CPU"
    assert report["model_name"]=="Mash Plus" and report["model_version"]=="V4"
    REPORTS[architecture]=report
    print("ARCHITECTURE_VALIDATION",architecture,report["selected_champion"]["type"],flush=True)

In [ ]:
# Prespecified architecture choice uses only validation, NEVER Dalmaj scores.
CANDIDATES=[]
for architecture,report in REPORTS.items():
    approved=[c for c in report["selected_champion"]["validation_challengers"] if c["passed_validation_gate"]]
    if approved:
        CANDIDATES.append((min(c["val_brier"] for c in approved),architecture))
PREFERRED_ARCH=min(CANDIDATES)[1] if CANDIDATES else None
SELECTION={"selection_rule":"lowest passing chronological validation Brier across two prespecified architectures; no spatial score used",
           "prespecified_candidates":[a for a,_ in TRIALS],
           "candidate_selected_on_validation":PREFERRED_ARCH,
           "selected_champion":"Mash Plus V4" if PREFERRED_ARCH else "PERSISTENCE_OR_SEASONAL_BASELINE",
           "independently_field_validated":False}
(OUT/"predeclared_validation_selection.json").write_text(json.dumps(SELECTION,indent=2))
print("VALIDATION_ONLY_NEURAL_WINNER",PREFERRED_ARCH or "NONE — BASELINE PREFERRED")

## 5 — Reveal Dalmaj test scores, then automated fail-closed assessment

No result from Dalmaj may be used to change training parameters in this experiment. Print wet-class F1 and positive support as well as Brier; high accuracy on predominantly dry pixels is misleading.

In [ ]:
sys.path.insert(0,str(ROOT/"ml"))
from judge_readiness import audit,markdown
for architecture,report in REPORTS.items():
    result=audit(report)
    # Other architecture is explicitly a prespecified comparator; it cannot be
    # presented as the overall neural winner when the validation choice differs.
    if PREFERRED_ARCH!=architecture and result["status"]=="PROMISING_PROXY_EXPERIMENT_NOT_FIELD_VALIDATED":
        result["status"]="NOT_SELECTED_BY_ARCHITECTURE_VALIDATION"
        result["satellite_proxy_performance_passes"]=False
        result["reasons"].append("A different predeclared architecture was selected on chronological validation")
    result_dir=OUT/"evaluations"/architecture
    (result_dir/"judge_readiness.json").write_text(json.dumps(result,indent=2,ensure_ascii=False))
    (result_dir/"judge_readiness.md").write_text(markdown(result))
    area=next(e for e in report["test_evaluations"] if e["region"]=="spatial_holdout")
    print("\nDALMAJ_RESULT",architecture,"valid_winner",report["selected_champion"]["type"],
          "SCIENCE",result["status"])
    print("SUPPORT",area["metrics"]["model"]["wet_truth_pixels"],
          "shared_fraction",area["shared_observed_fraction"])
    for name,metric in area["metrics"].items():
        print("METRIC",name,"Brier",metric.get("brier"),"WetF1",metric.get("f1"),
              "Recall",metric.get("recall"),"pixels",metric.get("evaluated_pixels"))
print("FIELD_VALIDATION: NOT DONE")

## 6 — Cross-sensor and blinded external-human reference preparation

Landsat is a separate sensor but still an **optical index proxy**, not a certified truth map. Its observation dates and 30-meter native pixels differ from Sentinel-2. If Landsat coverage/availability fails, record INCONCLUSIVE rather than inventing labels. Prepare blank georeferenced tasks for two independent reviewers; do not automatically fill their answers from machine predictions.

In [ ]:
CROSS_SENSOR={}
for architecture in REPORTS:
    result_dir=OUT/"evaluations"/architecture
    cross_out=result_dir/"landsat_cross_sensor.json"
    cmd=[sys.executable,str(ROOT/"ml/landsat_crosscheck.py"),
         "--spatial-manifest",str(PATHS["dalmaj_spatial_test"].parent/"satellite_manifest.json"),
         "--predictions",str(result_dir/"spatial_predictions_FOR_EXTERNAL_AUDIT.npz"),
         "--months","12","--out",str(cross_out)]
    try:
        subprocess.run(cmd,check=True)
        CROSS_SENSOR[architecture]=json.loads(cross_out.read_text())["status"]
    except Exception as exc:
        CROSS_SENSOR[architecture]="INCONCLUSIVE: "+type(exc).__name__+" (check execution log)"
        cross_out.write_text(json.dumps({"status":"INCONCLUSIVE_EXTERNAL_DATA_FAILURE",
                                         "error_type":type(exc).__name__,
                                         "independent_field_labels":False},indent=2))
    print("LANDSAT_SCREENING",architecture,CROSS_SENSOR[architecture])
subprocess.run([sys.executable,str(ROOT/"ml/prepare_field_review.py"),
                "--cube",str(PATHS["dalmaj_spatial_test"]),
                "--manifest",str(PATHS["dalmaj_spatial_test"].parent/"satellite_manifest.json"),
                "--out",str(OUT/"human_reference_review"),
                "--per-month","9","--last-months","12"],check=True)
print("HUMAN_LABELS_NOT_YET_COLLECTED")

## 7 — Export full evidence (including negative findings)

The output includes real Sentinel scene provenance, all original satellite arrays, a full archive of both neural contestants' validation and geographically held-out results, optional Landsat cross-sensor screening, and an **unlabeled** blind human-reference queue. Kaggle version status COMPLETE means cells finished, not that a model is recognized or ready for ecological deployment.

In [ ]:
SUMMARY={"model":"Mash Plus","version":"V4","source_commit":COMMIT,
         "source":"ESA Sentinel-2 (real); Landsat auxiliary sensor",
         "predeclared_aois":AOIS,"training_aois":["hawizeh_train","central_train","sinnaf_train"],
         "new_spatial_holdout":"dalmaj_spatial_test",
         "window":[START,END],"gpu":torch.cuda.get_device_name(0),
         "validation_selection":SELECTION,
         "cross_sensor_status":CROSS_SENSOR,
         "certified_reliable":False,"field_labels_verified":False,
         "experiment_results":{a:audit(r)["status"] for a,r in REPORTS.items()}}
(OUT/"experiment_summary.json").write_text(json.dumps(SUMMARY,indent=2,ensure_ascii=False))
archive=OUT/"Mash_Plus_V4_Real_Research.zip"
with zipfile.ZipFile(archive,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=3) as z:
    for item in OUT.rglob("*"):
        if item.is_file() and item!=archive:
            z.write(item,item.relative_to(OUT))
print("EVIDENCE_ZIP",archive,"BYTES",archive.stat().st_size,
      "SHA256",hashlib.sha256(archive.read_bytes()).hexdigest())
print("FINAL_VERDICT",SUMMARY["experiment_results"],"NOT_FIELD_VALIDATED")